# Flight Delay Prediction Using Machine Learning

**Month 5 – Data Science Internship**

This notebook documents the workflow for predicting whether a flight arrives more than 15 minutes late using the Airline Delay dataset.

**Target:** `DELAYED = 1` when `ARRIVAL_DELAY > 15`; otherwise `DELAYED = 0`. Cancelled and diverted flights are excluded during feature engineering.

Run cells in order. The dataset is large, so processing may take time and require substantial disk space.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

## 2. Verify dataset files

Place `flights.csv`, `airlines.csv`, and `airports.csv` in `data/raw/`.

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
required_files = [
    RAW_DATA_DIR / "flights.csv",
    RAW_DATA_DIR / "airlines.csv",
    RAW_DATA_DIR / "airports.csv"
]
missing_files = [p.name for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing files in {RAW_DATA_DIR}: {missing_files}")

for path in required_files:
    print(f"{path.name}: {path.stat().st_size / (1024 ** 2):.2f} MB")

## 3. Data understanding

The module uses chunk-based reading for large-file statistics.

In [ ]:
from data_understanding import analyze_datasets
understanding_results = analyze_datasets()

## 4. Data preprocessing

This stage applies the configured missing-value threshold, removes duplicates within each chunk, and writes the processed dataset. It does not guarantee global duplicate removal across chunk boundaries.

In [ ]:
from data_preprocessing import preprocess_flight_data
processed_dataset_path = preprocess_flight_data()
print(processed_dataset_path)

## 5. Feature engineering

Cancelled and diverted flights are excluded. The target and scheduled calendar/time features are created, and post-flight information is removed to reduce leakage.

In [ ]:
from feature_engineering import engineer_features
featured_dataset_path = engineer_features()
print(featured_dataset_path)

## 6. Inspect model-ready data

Load a sample rather than the complete multi-million-row dataset.

In [ ]:
featured_sample = pd.read_csv(featured_dataset_path, nrows=10000, low_memory=False)
print("Sample shape:", featured_sample.shape)
display(featured_sample.head())
display(featured_sample.dtypes.to_frame("Data Type"))

## 7. Model training

The training module draws a reproducible sample capped at 300,000 records and uses a stratified 80/20 train-test split.

**Environment note:** If Scikit-learn raises a Windows DLL or Application Control error, stop and resolve the interpreter/security-policy issue before continuing.

In [ ]:
from model_training import run_model_training
training_results = run_model_training()
print("Training records:", f"{training_results['training_records']:,}")
print("Testing records:", f"{training_results['testing_records']:,}")

## 8. Model evaluation

Compare accuracy, precision, recall, F1-score, and ROC-AUC. The current implementation selects the highest F1-score, using ROC-AUC as a tie-breaker.

In [ ]:
from model_evaluation import evaluate_models
evaluation_results = evaluate_models(
    models=training_results["models"],
    X_test=training_results["X_test"],
    y_test=training_results["y_test"]
)
display(evaluation_results["comparison"])
print("Selected model:", evaluation_results["best_model_name"])

## 9. Review generated artifacts

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
for path in sorted([p for p in OUTPUT_DIR.glob("*") if p.is_file()] +
                   [p for p in MODEL_DIR.glob("*") if p.is_file()]):
    print(path.relative_to(PROJECT_ROOT))

## 10. Results and interpretation

Evaluation was performed on the stratified holdout set of **57,140 flights** (46,936 on-time and 10,204 delayed) from the reproducible training sample. The positive class is a flight delayed by more than 15 minutes.

| Metric | Logistic Regression | Random Forest |
|---|---:|---:|
| Accuracy | 0.5936 | **0.6158** |
| Precision (delayed) | 0.2445 | **0.2551** |
| Recall (delayed) | **0.6105** | 0.6000 |
| F1-score (delayed) | 0.3492 | **0.3580** |
| ROC-AUC | 0.6382 | **0.6544** |

### Confusion matrices

Rows are actual classes and columns are predicted classes.

| Model | True negatives | False positives | False negatives | True positives |
|---|---:|---:|---:|---:|
| Logistic Regression | 27,686 | 19,250 | 3,974 | 6,230 |
| Random Forest | 29,062 | 17,874 | 4,082 | 6,122 |

Both models identify about 60% of delayed flights, but miss 3,974 (Logistic Regression) and 4,082 (Random Forest) delays. Precision is low for both: of flights flagged as delayed, approximately 24% (Logistic Regression) and 26% (Random Forest) were actually delayed, so false positives are common.

Random Forest is selected because it has the higher F1-score (0.3580 vs. 0.3492), with higher accuracy, precision, and ROC-AUC. Logistic Regression has slightly higher delayed-flight recall (0.6105 vs. 0.6000), detecting 108 more delayed flights in this holdout set at the cost of more false positives. The ROC-AUC values (0.6382 and 0.6544) indicate only modest discrimination above the 0.50 random-ranking baseline.

These results describe this sampled, stratified random holdout set, not future operations or all flights. The split is not chronological, and model performance may differ across time, airlines, routes, and airports.

## 11. Conclusion

This project implements an end-to-end workflow for flight-delay classification, from raw-data inspection and preprocessing through feature engineering, model training, and evaluation. On the 57,140-flight stratified holdout set, Random Forest was selected for its higher F1-score (0.3580), accuracy (0.6158), and ROC-AUC (0.6544). Logistic Regression had slightly higher recall for delayed flights (0.6105 versus 0.6000).

Both models identify about 60% of delayed flights, but their delayed-class precision is low (0.2445 for Logistic Regression and 0.2551 for Random Forest), and each misses roughly 40% of actual delays. Their current results therefore show some ability to distinguish delays, but should not be treated as sufficient for operational decisions without further validation.

These results are limited to a reproducible sample capped at 300,000 records and a stratified random split, which does not test performance on future time periods. Before deployment, evaluate on a chronological holdout, consider probability-threshold tuning based on the cost of missed delays versus false alerts, and investigate additional signals such as weather along with performance across airlines and routes.